##### make_interval()

- is used to create an **INTERVAL type** from individual components like **years, months, weeks, days, hours, minutes, and seconds**.
- It is available starting from **Spark version 3.5.0.**
- **Unspecified arguments** are **defaulted to 0**.
- If you provide **no arguments** the result is an **INTERVAL with 0 seconds**.

##### Syntax

     make_interval(years=<years>, months=<months>, weeks=<weeks>, days=<days>, hours=<hours>, mins=<mins>, secs=<secs>)

     # Correct order
     make_interval(years, months, weeks, days, hours, mins, secs)

##### Parameters

| parameter |          Description                              |
|-----------|---------------------------------------------------|
|   years   | The number of years, positive or negative.        |
|   months  | The number of months, positive or negative.       |
|   weeks   | The number of weeks, positive or negative.        |
|   days    | The number of days, positive or negative.         |
|   hours   | The number of hours, positive or negative.        |
|   mins    | The number of minutes, positive or negative.      |
|   secs    | The number of seconds, with fractional parts in microsecond precision. | 

**Return**
- A **new column** that contains an **interval**.

In [0]:
from pyspark.sql.functions import lit, col, expr, make_interval, to_date, add_months

##### 1) Create an Interval from `Years, Months and Days`

      make_interval(
          lit(1),      # years
          lit(2),      # months
          lit(3),      # weeks (7x3=21 days)
          lit(4),      # days  21 + 4 = 25 days
          lit(5),      # hours
          lit(6),      # minutes
          lit(7.123)   # seconds (can be decimal)
        )

In [0]:
# Create a DataFrame with a single row
df = spark.range(1)
# Create an interval from 1 year, 2 months, and 3 days (weeks parameter is third, not days)
df_01 = df.withColumn("interval", make_interval(lit(1), lit(2), lit(0), lit(3)))

# Display the result
display(df_01)

id,interval
0,1 years 2 months 3 days


In [0]:
# Create a DataFrame with a single row
df = spark.range(1)
# Create an interval from 1 year, 2 months, and 3 days (weeks parameter is third, not days)
df_02 = df.withColumn("interval", make_interval(lit(1), lit(2), lit(5), lit(3)))

# Display the result
display(df_02)

id,interval
0,1 years 2 months 38 days


In [0]:
# Create a DataFrame with a single row
df = spark.range(1)
# Create an interval from 1 year, 2 months, and 3 days (weeks parameter is third, not days)
df_03 = df.withColumn("interval", make_interval(lit(1), lit(2), lit(3)))

# Display the result
display(df_03)

id,interval
0,1 years 2 months 21 days


##### 2) Create an Interval from `Years and Months`

In [0]:
# Create a DataFrame with a single row
df = spark.range(1)
# Create an interval from 1 year and 2 months
df_02 = df.withColumn("interval", make_interval(lit(1), lit(2), lit(0)))

# Display the result
display(df_02)

id,interval
0,1 years 2 months


##### 3) Create an Interval from `Days`

In [0]:
# Create a DataFrame with a single row
df = spark.range(1)
# Create an interval from 3 days
df_03 = df.withColumn("interval", make_interval(lit(0), lit(0), lit(3)))

# Display the result
display(df_03)

id,interval
0,21 days


##### 4) Add an Interval to a `Date`

In [0]:
# Create a DataFrame with a date column
df = spark.createDataFrame([(1,)], ["id"]).withColumn("date", to_date(lit("2022-01-01")))
display(df)

# Add the interval to the date column
df_04 = df.withColumn("new_date", expr("date + make_interval(1, 2, 0)"))
# Display the result
display(df_04)

id,date
1,2022-01-01


id,date,new_date
1,2022-01-01,2023-03-01


##### 5) Create an Interval from `Years, Months, Week, Days, Hour, Minute, Second`

In [0]:
df1 = spark.range(1)

df1.select(
    make_interval(
        lit(1),      # years
        lit(2),      # months
        lit(3),      # weeks (7x3=21 days)
        lit(4),      # days  21 + 4 = 25 days
        lit(5),      # hours
        lit(6),      # minutes
        lit(7.123)   # seconds (can be decimal)
    ).alias("interval_value")
).display()

interval_value
1 years 2 months 25 days 5 hours 6 minutes 7.123 seconds


In [0]:
df = spark.createDataFrame(
    [[100, 11, 1, 1, 12, 30, 1.001001]],
    ['year', 'month', 'week', 'day', 'hour', 'min', 'sec']
)

df_05 = df.select("*",
    make_interval("year", "month", "week", "day", "hour", "min", "sec").alias("full_interval"),
    make_interval("year", "month", "week", "day", "hour", "min").alias("no_sec"),
    make_interval("year", "month", "week", "day", "hour").alias("no_min_sec"),
    make_interval("year", "month", "week", "day").alias("no_time"),
    make_interval("year", "month", "week").alias("no_day_time"),
    make_interval("year", "month").alias("year_month"),
    make_interval("year").alias("only_year")
)

display(df_05)

year,month,week,day,hour,min,sec,full_interval,no_sec,no_min_sec,no_time,no_day_time,year_month,only_year
100,11,1,1,12,30,1.001001,100 years 11 months 8 days 12 hours 30 minutes 1.001001 seconds,100 years 11 months 8 days 12 hours 30 minutes,100 years 11 months 8 days 12 hours,100 years 11 months 8 days,100 years 11 months 7 days,100 years 11 months,100 years


In [0]:
spark.range(1).select(make_interval().alias("No_Argument")).display()

No_Argument
0 seconds


##### 6) Create an interval with a mix of components
- If you provide **no arguments** the result is an `INTERVAL` with **0 seconds**.
- **Unspecified arguments** are **defaulted to 0**. 

In [0]:
%sql
SELECT make_interval(100, 11)
UNION ALL
SELECT make_interval(100, null)
UNION ALL
SELECT make_interval()
UNION ALL
SELECT make_interval(0, 0, 1, 1, 12, 30, 01.001001);

"make_interval(100,11)"
100 years 11 months
null
0 seconds
8 days 12 hours 30 minutes 1.001001 seconds
